In [2]:
from fileformer import utils, Decoder, Encoder

In [3]:
import torch
from flax import nnx
from torchinfo import summary

In [4]:
decoder = Decoder(258, 256, 2, 2, 'cpu', 256, 0.0, 256, 'gelu')

In [5]:
summary(decoder)

Layer (type:depth-idx)                        Param #
Decoder                                       --
├─Embedding: 1-1                              66,048
├─RotaryPositionalEmbeddings: 1-2             --
├─ModuleList: 1-3                             --
│    └─DecoderLayer: 2-1                      --
│    │    └─MultiHeadLinearAttention: 3-1     263,168
│    │    └─MLP: 3-2                          131,584
│    │    └─LayerNorm: 3-3                    512
│    │    └─LayerNorm: 3-4                    512
│    │    └─Dropout: 3-5                      --
│    └─DecoderLayer: 2-2                      --
│    │    └─MultiHeadLinearAttention: 3-6     263,168
│    │    └─MLP: 3-7                          131,584
│    │    └─LayerNorm: 3-8                    512
│    │    └─LayerNorm: 3-9                    512
│    │    └─Dropout: 3-10                     --
├─Dropout: 1-4                                --
├─Linear: 1-5                                 66,306
Total params: 923,906
Trainable 

In [6]:
from fileformer import ENWIK8Dataset
from fileformer.tokenizer import ByteLevelTokenizer
import torch
from torch import Tensor
from torch.utils.data import DataLoader

In [7]:
dataset = ENWIK8Dataset('av601.jpg', ByteLevelTokenizer(), 512, 0, "cache")
loader = DataLoader(dataset, batch_size=4, shuffle=False)

In [8]:
x = next(iter(loader))

In [9]:
decoder.eval()

Decoder(
  (chunk_emb): Embedding(258, 256)
  (pe): RotaryPositionalEmbeddings()
  (layers): ModuleList(
    (0-1): 2 x DecoderLayer(
      (self_attention): MultiHeadLinearAttention(
        (Q_layer): Linear(in_features=256, out_features=256, bias=True)
        (K_layer): Linear(in_features=256, out_features=256, bias=True)
        (V_layer): Linear(in_features=256, out_features=256, bias=True)
        (fc_out): Linear(in_features=256, out_features=256, bias=True)
      )
      (mlp): MLP(
        (activation): GELU(approximate='none')
        (mlp): Sequential(
          (0): Linear(in_features=256, out_features=256, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=256, out_features=256, bias=True)
          (3): Dropout(p=0.0, inplace=False)
        )
      )
      (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (norm3): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )

In [10]:
x, pad, cus = x

In [11]:
pad = pad.to(torch.bool)

In [12]:
x

tensor([[257, 218, 257,  ..., 118,  40,  41],
        [ 56,  87, 134,  ..., 227,  94, 135],
        [ 68,  57,  99,  ..., 159, 154, 240],
        [169, 185,  32,  ..., 214, 130,  32]])

In [13]:
decoder(x, pad).shape

torch.Size([4, 512, 258])